
# Word2Vec: Distributed Word Representations

**Deep Learning Course (B.Sc. Computer Science)**  
Course website: https://fum-cs.github.io/deep-learning

---

# Learning Objectives

After completing this notebook, students should be able to:

1. Explain the limitations of one-hot encoding.
2. Describe the distributional hypothesis.
3. Explain Word2Vec as a shallow neural network.
4. Relate Word2Vec to encoder–decoder architectures.
5. Distinguish between Skip-Gram and CBOW.
6. Generate Word2Vec training examples from text.
7. Train Word2Vec using Gensim.
8. Analyze learned embeddings through similarity and analogy tasks.
9. Visualize embeddings using t-SNE.
10. Compare Word2Vec with autoencoders.
11. Understand how Word2Vec relates to modern transformer embeddings.



# Session Roadmap

1. Motivation
2. One-Hot Encoding vs Embeddings
3. Distributional Hypothesis
4. Word2Vec Architecture
5. Skip-Gram and CBOW
6. Preparing Training Data
7. Training on *Anne of Green Gables*
8. Exploring Embeddings
9. Analogies
10. Visualization with t-SNE
11. spaCy Embeddings
12. Autoencoders vs Word2Vec
13. Word2Vec vs Transformer Embeddings
14. Exercises and Discussion Questions



# 1. Why Do We Need Word Embeddings?

Consider the vocabulary:

```text
cat, dog, horse, computer
```

Using one-hot encoding:

```text
cat      = [1,0,0,0]
dog      = [0,1,0,0]
horse    = [0,0,1,0]
computer = [0,0,0,1]
```

Problems:

- Very high dimensional
- Sparse
- No semantic information
- Similar words are not close to each other

Word embeddings solve these problems by learning dense vectors.


In [1]:

import numpy as np

vocab = ["cat","dog","horse","computer"]

for i, word in enumerate(vocab):
    vec = np.zeros(len(vocab))
    vec[i] = 1
    print(word, vec)


cat [1. 0. 0. 0.]
dog [0. 1. 0. 0.]
horse [0. 0. 1. 0.]
computer [0. 0. 0. 1.]



# 2. Distributional Hypothesis

The foundation of Word2Vec:

> You shall know a word by the company it keeps.
>
> — J.R. Firth

Words occurring in similar contexts tend to have similar meanings.

Example:

```text
The cat chased the mouse.
The dog chased the ball.
```

The words *cat* and *dog* appear in similar contexts and therefore receive similar vector representations.



# Discussion Question 1

Why would the words **doctor** and **hospital** often have related embeddings even though they refer to different concepts?



# 3. Word2Vec as an Encoder–Decoder Model

In Lecture 8 we studied encoder–decoder architectures.

Word2Vec can be viewed as a simplified encoder–decoder network.

## Encoder

Input:

```text
one-hot word
```

Output:

```text
dense embedding vector
```

The embedding matrix acts as an encoder.

## Decoder

Input:

```text
embedding vector
```

Output:

```text
probability distribution over vocabulary
```

The decoder predicts context words.

The latent embedding corresponds to the bottleneck representation found in autoencoders.



# Autoencoder Analogy

Autoencoder:

```text
Input
 ↓
Encoder
 ↓
Latent Space
 ↓
Decoder
 ↓
Reconstructed Input
```

Word2Vec:

```text
Word
 ↓
Encoder (Embedding Matrix)
 ↓
Latent Vector
 ↓
Decoder
 ↓
Context Prediction
```

Key difference:

- Autoencoder reconstructs the input.
- Word2Vec predicts surrounding context.



# Discussion Question 2

Why might context prediction produce richer semantic representations than simply reconstructing the same input word?



# 4. Skip-Gram

Skip-Gram predicts surrounding words from a center word.

Sentence:

```text
Anne was beginning to get very tired
```

Window size = 2

Center word:

```text
beginning
```

Context:

```text
Anne, was, to, get
```

Training pairs:

```text
(beginning → Anne)
(beginning → was)
(beginning → to)
(beginning → get)
```



# 5. CBOW

CBOW performs the reverse task.

Input:

```text
Anne, was, to, get
```

Target:

```text
beginning
```

Comparison:

| Property | Skip-Gram | CBOW |
|----------|-----------|------|
| Training speed | Slower | Faster |
| Rare words | Better | Worse |
| Small corpora | Better | Often weaker |
| Large corpora | Good | Excellent |



# Exercise 1

Using the sentence

```text
Matthew drove Anne to Green Gables
```

and a window size of 2:

1. Create all Skip-Gram pairs.
2. Create all CBOW examples.



# 6. Preparing Training Examples

The most important practical concept in Word2Vec is understanding how the sliding window generates training examples.


In [2]:

sentence = "Anne was beginning to get very tired".split()

window_size = 2

for i, center_word in enumerate(sentence):

    start = max(0, i-window_size)
    end = min(len(sentence), i+window_size+1)

    context = [
        sentence[j]
        for j in range(start, end)
        if j != i
    ]

    print(center_word, "->", context)


Anne -> ['was', 'beginning']
was -> ['Anne', 'beginning', 'to']
beginning -> ['Anne', 'was', 'to', 'get']
to -> ['was', 'beginning', 'get', 'very']
get -> ['beginning', 'to', 'very', 'tired']
very -> ['to', 'get', 'tired']
tired -> ['get', 'very']



# Exercise 2

Change the window size to:

- 1
- 3
- 5

How does the number of training examples change?



# 7. Loading Anne of Green Gables


In [3]:

from pathlib import Path
import re

corpus_path = Path("data/Anne_of_Green_Gables.txt")

text = corpus_path.read_text(
    encoding="utf-8"
)

text = text.lower()
text = re.sub(r"[^a-z\s]", " ", text)

tokens = text.split()

print("Number of tokens:", len(tokens))


Number of tokens: 107178



# Inspecting the Corpus


In [4]:

from collections import Counter

counter = Counter(tokens)

counter.most_common(20)


[('the', 3926),
 ('and', 3398),
 ('i', 3265),
 ('to', 3046),
 ('a', 2229),
 ('it', 2099),
 ('of', 1923),
 ('you', 1702),
 ('she', 1519),
 ('in', 1479),
 ('that', 1371),
 ('was', 1364),
 ('her', 1315),
 ('anne', 1214),
 ('t', 1189),
 ('s', 1153),
 ('marilla', 851),
 ('but', 849),
 ('be', 822),
 ('as', 791)]


# Discussion Question 3

Why are common words such as:

```text
the, and, of, to
```

usually less informative than content words?



# 8. Training Word2Vec with Gensim


In [5]:

from gensim.models import Word2Vec

sentences = [tokens]

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,
    epochs=10
)


c:\Users\m.amintoosi\.conda\envs\pth-gpu\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



# Model Parameters

Important hyperparameters:

- vector_size
- window
- min_count
- epochs
- sg

Experimenting with these values is often more important than changing the neural architecture itself.



# Exercise 3

Train three models:

1. vector_size=50
2. vector_size=100
3. vector_size=300

Compare the results.



# 9. Exploring Learned Embeddings


In [6]:

model.wv.most_similar("anne", topn=10)


[('surprised', 0.9965387582778931),
 ('concert', 0.9963647127151489),
 ('confession', 0.9953275322914124),
 ('invited', 0.9946743249893188),
 ('school', 0.9944130778312683),
 ('life', 0.9940248727798462),
 ('tea', 0.9940096735954285),
 ('lily', 0.9939918518066406),
 ('history', 0.993578314781189),
 ('interest', 0.9934440851211548)]

In [7]:

model.wv.most_similar("marilla", topn=10)


[('herself', 0.993959903717041),
 ('head', 0.9935182332992554),
 ('door', 0.993461549282074),
 ('sorrel', 0.9931078553199768),
 ('came', 0.9930408000946045),
 ('mare', 0.9927841424942017),
 ('buggy', 0.9926896095275879),
 ('set', 0.992451548576355),
 ('corner', 0.9923194646835327),
 ('reverie', 0.9923020005226135)]

In [8]:

model.wv.similarity("anne", "marilla")


0.98570067

In [9]:

model.wv.similarity("anne", "matthew")


0.9744315


# Reflection

Do the most similar words appear reasonable?

If not, what characteristics of the corpus might explain the results?



# 10. Word Analogies

One of the most famous discoveries about Word2Vec is vector arithmetic.

The classic example:

```text
king − man + woman ≈ queen
```

Although the Anne corpus is much smaller than Wikipedia, we can still experiment with analogies.


In [10]:

model.wv.most_similar(
    positive=["woman","king"],
    negative=["man"],
    topn=10
)


KeyError: "Key 'king' not present in vocabulary"


# Analogy Activities

Try:

```python
model.wv.most_similar(
    positive=[A,C],
    negative=[B]
)
```

where:

- A is to B as C is to ?

Examples:

1. paris : france :: tehran : ?
2. king : queen :: man : ?
3. walk : walking :: swim : ?



# Exercise 4

Design three additional analogy questions and test them.



# 11. Visualizing Embeddings with t-SNE

Word vectors typically live in 100–300 dimensions.

t-SNE projects them into two dimensions for visualization.


In [ ]:

import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

words = model.wv.index_to_key[:80]

vectors = np.array(
    [model.wv[w] for w in words]
)

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=10
)

coords = tsne.fit_transform(vectors)

plt.figure(figsize=(12,10))

for word, (x,y) in zip(words, coords):
    plt.scatter(x,y)
    plt.annotate(word,(x,y))

plt.title("t-SNE Visualization of Word Embeddings")
plt.show()



# t-SNE Investigation Activity

Look for:

- character clusters
- location clusters
- emotion-related words
- family-related words

Questions:

1. Which words appear unexpectedly close?
2. Which words appear unexpectedly far apart?
3. Does semantic structure emerge?



# Exercise 5

Create a visualization using only character names.

Possible candidates:

- Anne
- Marilla
- Matthew
- Diana
- Gilbert



# 12. Using Pretrained spaCy Embeddings


In [15]:
# !pip install spacy
# %python -m spacy download en_core_web_md

In [16]:
import spacy

nlp = spacy.load("en_core_web_md")


In [17]:

doc1 = nlp("cat")
doc2 = nlp("dog")
doc3 = nlp("car")

print(doc1.similarity(doc2))
print(doc1.similarity(doc3))


1.0000000568192473
0.19304996123900908


In [18]:

words = [
    "teacher",
    "student",
    "school",
    "computer"
]

for word in words:
    token = nlp.vocab[word]
    print(word, token.vector.shape)


teacher (300,)
student (300,)
school (300,)
computer (300,)



# Gensim vs spaCy

| Gensim | spaCy |
|---------|--------|
| Train embeddings | Usually uses pretrained embeddings |
| Educational | Production NLP |
| Corpus-specific | General-purpose |
| Full control | Convenient |



# Exercise 6

Compare the similarity scores:

```text
cat vs dog
cat vs tiger
cat vs computer
```

using spaCy.



# 13. Word2Vec and Autoencoders

Both methods learn latent representations.

| Feature | Autoencoder | Word2Vec |
|----------|------------|----------|
| Input | Data sample | Word |
| Target | Original input | Context |
| Encoder | Neural network | Embedding matrix |
| Decoder | Neural network | Softmax predictor |
| Output | Reconstruction | Context probabilities |
| Goal | Compression | Semantic representation |



# Conceptual Exercise

Suppose we train:

1. An autoencoder on images of cats and dogs.
2. Word2Vec on a text corpus about cats and dogs.

What kind of information will each latent representation capture?



# 14. From Word2Vec to Transformers

Word2Vec was a major breakthrough, but modern NLP systems typically use transformer-based embeddings.

Examples:

- BERT
- RoBERTa
- GPT embeddings
- Sentence Transformers

The key difference is context.



# Static vs Contextual Embeddings

Word2Vec:

```text
bank → one vector
```

regardless of context.

Transformer:

```text
bank (river bank)
bank (financial bank)
```

receives different embeddings.

This is called a contextual embedding.



# Example

Sentence 1:

```text
I deposited money in the bank.
```

Sentence 2:

```text
The fisherman sat by the bank.
```

Word2Vec:

```text
same vector
```

Transformer:

```text
different vectors
```



# Discussion Question 4

Why are contextual embeddings particularly useful for machine translation and question answering?



# Mini Research Activity

Read about:

- Word2Vec
- GloVe
- FastText
- BERT

Create a table comparing:

- architecture
- context awareness
- training objective
- strengths



# Summary

In this notebook you learned:

- One-hot encoding limitations
- Distributional hypothesis
- Skip-Gram and CBOW
- Word2Vec as a simple encoder–decoder model
- Construction of training examples
- Training Word2Vec using Gensim
- Similarity and analogy tasks
- t-SNE visualization
- spaCy embeddings
- Connections with autoencoders
- Evolution from Word2Vec to transformer embeddings

Word2Vec remains one of the most influential ideas in deep learning because it demonstrated that useful semantic structure can emerge from self-supervised prediction tasks.
